# CreditLens Stage 2: train-only EDA

이 노트북은 재현 가능한 Stage 2 EDA 실행 진입점입니다. 모든 피처 통계는 `train` 고객만 사용하며 validation과 test는 분할 무결성 확인 외에는 열지 않습니다. 최종 설명은 LLM이 읽기 쉬운 `docs/Stage2_EDA_Report.md`에 저장됩니다.

## 실행 전 확인

먼저 프로젝트 루트에서 고객 분할을 생성합니다.

```bash
PYTHONPATH=src .venv/bin/python -m creditlens.data.split_data
```

고객별 분할표는 `data/interim/`에만 저장되고 Git에서 제외됩니다.

In [ ]:
from pathlib import Path
import sys

project_root = Path.cwd()
if project_root.name == 'notebooks':
    project_root = project_root.parent
sys.path.insert(0, str(project_root / 'src'))

assignment_path = project_root / 'data/interim/customer_splits.csv'
assert assignment_path.is_file(), '먼저 Stage 2 고객 분할 명령을 실행하세요.'

## train-only EDA 실행

아래 함수는 분할표와 원본의 고객·TARGET 계약을 검증한 뒤 train 행만 통계 메모리에 적재합니다. 모델 학습이나 전처리는 수행하지 않습니다.

In [ ]:
from creditlens.analysis.stage2_eda import run_stage2_eda

eda_result = run_stage2_eda(
    application_path=project_root / 'data/raw/application_train.csv',
    assignment_path=assignment_path,
    json_path=project_root / 'reports/stage2_eda.json',
    report_path=project_root / 'docs/Stage2_EDA_Report.md',
    figures_dir=project_root / 'reports/figures',
)
eda_result['analysis_scope']

## 핵심 결과 확인

상세 수치와 해석은 `docs/Stage2_EDA_Report.md`, 전체 집계는 `reports/stage2_eda.json`을 확인합니다. 고객 ID와 행 단위 금융정보는 두 산출물에 포함되지 않습니다.

In [ ]:
import pandas as pd

pd.DataFrame(
    {
        '값': [
            eda_result['analysis_scope']['train_rows'],
            eda_result['analysis_scope']['feature_columns'],
            eda_result['target_distribution']['1']['rate'],
            eda_result['column_summary']['columns_with_missing'],
        ]
    },
    index=['train 고객', '피처 수', 'TARGET=1 비율', '결측 피처 수'],
)